# 02. Data Quality Assessment

This notebook performs basic data-quality checks on the source datasets:

- Missing values
- Exact duplicate rows
- Primary-key completeness
- Primary-key uniqueness

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"

print(f"Raw data directory: {raw_dir}")

Raw data directory: /home/pamern/Projects/fashion-ecommerce-analytics/data/raw


In [3]:
table_results = []
column_results = []

for file_path in sorted(raw_dir.glob("*.csv")):
    df = pd.read_csv(file_path, low_memory=False)

    # Kiểm tra tổng quan từng bảng
    table_results.append({
        "table_name": file_path.stem,
        "rows": len(df),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

    # Kiểm tra missing value theo từng cột
    for column in df.columns:
        null_count = int(df[column].isna().sum())

        column_results.append({
            "table_name": file_path.stem,
            "column_name": column,
            "data_type": str(df[column].dtype),
            "null_count": null_count,
            "null_percent": round(null_count / len(df) * 100, 2),
            "unique_values": df[column].nunique(dropna=True),
        })


table_quality = pd.DataFrame(table_results)
column_quality = pd.DataFrame(column_results)

In [4]:
table_quality

,table_name,rows,missing_cells,duplicate_rows
0,customers,121930,0,0
1,geography,39948,0,0
2,inventory,60247,0,0
3,order_items,714669,1152816,0
4,orders,646945,0,0
5,payments,646945,0,0
6,products,2412,0,0
7,promotions,50,40,0
8,returns,39939,0,0
9,reviews,113551,0,0


In [5]:
columns_with_missing = (
    column_quality[column_quality["null_count"] > 0]
    .sort_values(
        by="null_percent",
        ascending=False,
    )
    .reset_index(drop=True)
)

columns_with_missing

,table_name,column_name,data_type,null_count,null_percent,unique_values
0,order_items,promo_id_2,str,714463,99.97,2
1,promotions,applicable_category,str,40,80.00,2
2,order_items,promo_id,str,438353,61.34,50
